# Training

In [2]:
import os
import sys
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import webdataset as wds
import json
import time
import csv
import random
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. HARDWARE & PATH INITIALIZATION
# ==========================================
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt (in that order).
    Usage: python train_model.py --gpu 1   OR   GPU_ID=1 python train_model.py
    """
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

GPU_ID = select_gpu()
torch.cuda.set_device(GPU_ID)
device = torch.device(f"cuda:{GPU_ID}")

if torch.cuda.is_available():
    print(f"[SYSTEM] PyTorch initialized. Active Device: {torch.cuda.get_device_name(GPU_ID)} (cuda:{GPU_ID})")
    print(f"[SYSTEM] CUDA Version: {torch.version.cuda} | PyTorch Version: {torch.__version__}")
    print(f"[SYSTEM] GPU Memory: {torch.cuda.get_device_properties(GPU_ID).total_memory / 1024**3:.1f} GB")
else:
    print(f"[WARNING] PyTorch could not find a CUDA device. Falling back to CPU.")

# --- Reproducibility ---
MASTER_SEED = 42
torch.manual_seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
random.seed(MASTER_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(MASTER_SEED)
    torch.cuda.manual_seed_all(MASTER_SEED)
torch.backends.cudnn.deterministic = True
# FIX #5: Enable cuDNN auto-tuner — safe because input shapes are fixed.
# For maximum speed, also set deterministic=False above.
torch.backends.cudnn.benchmark = True
print(f"[SYSTEM] All RNG sources seeded with MASTER_SEED={MASTER_SEED}")
print(f"[SYSTEM] cudnn.deterministic={torch.backends.cudnn.deterministic}, cudnn.benchmark={torch.backends.cudnn.benchmark}")

user_prefix = input("\nEnter version prefix (e.g., tcn_v1) [Press Enter for default 'unknown_version']: ").strip()
VERSION_PREFIX = user_prefix if user_prefix else "unknown_version"

PROJECT_DIR = Path(os.path.expanduser("~/Capstone/Implementation"))
LOCAL_SHARDS_DIR = PROJECT_DIR / "data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards"
SAVE_DIR = PROJECT_DIR / f"models/{VERSION_PREFIX}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SCALARS_PATH = LOCAL_SHARDS_DIR / "normalization_scalars.json"

prefix = "/home/sanke24839/"

print(f"\n[PATHS] PROJECT_DIR     : {str(PROJECT_DIR).replace(prefix, '')}")
print(f"[PATHS] LOCAL_SHARDS_DIR: {str(LOCAL_SHARDS_DIR).replace(prefix, '')}")
print(f"[PATHS] SAVE_DIR        : {str(SAVE_DIR).replace(prefix, '')}")
print(f"[PATHS] SCALARS_PATH    : {str(SCALARS_PATH).replace(prefix, '')}")

with open(SCALARS_PATH, 'r') as f:
    scalars = json.load(f)

print(f"\n[DATA] Normalization scalars loaded successfully.")
print(f"[DATA] Feature Order: {scalars['feature_order']}")
print(f"[DATA] Number of Features: {len(scalars['feature_order'])}")

for i, feat in enumerate(scalars['feature_order']):
    print(f"[DATA]   [{i}] {feat:>30s} | min={scalars['global_minimums'][feat]:>12.4f} | max={scalars['global_maximums'][feat]:>12.4f}")

mins = torch.tensor([scalars['global_minimums'][c] for c in scalars['feature_order']], device=device)
maxs = torch.tensor([scalars['global_maximums'][c] for c in scalars['feature_order']], device=device)
ranges = maxs - mins
ranges[ranges == 0] = 1.0

print(f"\n[DATA] mins   tensor: shape={mins.shape}, dtype={mins.dtype}, device={mins.device}")
print(f"[DATA]   values: {mins.cpu().numpy()}")
print(f"[DATA] maxs   tensor: shape={maxs.shape}, dtype={maxs.dtype}, device={maxs.device}")
print(f"[DATA]   values: {maxs.cpu().numpy()}")
print(f"[DATA] ranges tensor: shape={ranges.shape}")
print(f"[DATA]   values: {ranges.cpu().numpy()}")

# ==========================================
# 2. ARCHITECTURE & DATALOADERS
# ==========================================
DEFAULT_EMPIRICAL = {
    "C0": 143.22, "C1": 140.25, "h0": 4.8713, "h1": 5.3871,
    "k01": 0.078038, "k10": 0.028120, "q0": 35.0, "q1": 35.0,
    "T_amb": 30.5
}

DEFAULT_FEATURE_INDICES = {
    "P0": 0, "P1": 1, "T0": 4, "T1": 5
}

BATCH_SIZE = 16384
NUM_WORKERS = 8
WINDOW_SIZE = 95
STRIDE = 10

# Corrected DT from actual timestamp intervals in the dataset
# Measured intervals: 0.109, 0.107, 0.114, 0.108, 0.112, 0.108, 0.113, 0.108, 0.112
# Mean ≈ 0.11s
DT = 0.11

print(f"\n[CONFIG] BATCH_SIZE={BATCH_SIZE}, NUM_WORKERS={NUM_WORKERS}")
print(f"[CONFIG] WINDOW_SIZE={WINDOW_SIZE}, STRIDE={STRIDE}")
print(f"[CONFIG] DT={DT} (corrected from actual timestamp intervals)")
print(f"[CONFIG] Feature Indices: {DEFAULT_FEATURE_INDICES}")

print(f"\n[PHYSICS] Empirical parameter values (from prior calculation):")
for k, v in DEFAULT_EMPIRICAL.items():
    if k in ('q0', 'q1'):
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (unbounded via softplus, init guess)")
    elif k == 'T_amb':
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (bounds: [25.0, 35.0])")
    else:
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (bounds: [{v*0.75:.6f}, {v*1.25:.6f}])")


# FIX #2: Expanded TCN from [32, 64, 128] to [32, 64, 128, 256, 512]
# Receptive field: 1 + (3-1)*(1+2+4+8+16) = 63 timesteps (covers full 50-step input)
# Parameter count: ~527K (up from ~32K) — GPU stays compute-bound
class DilatedTCN(nn.Module):
    def __init__(self, num_inputs=6, num_channels=[32, 64, 128, 256, 512], kernel_size=3):
        super(DilatedTCN, self).__init__()
        layers = []
        in_channels = num_inputs
        for i, out_channels in enumerate(num_channels):
            dilation_size = 2 ** i
            padding = (kernel_size - 1) * dilation_size
            layers += [
                nn.ConstantPad1d((padding, 0), 0.0),
                nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
            ]
            in_channels = out_channels
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.tcn(x)
        return self.fc(out[:, :, -1])


class PINNEngine(nn.Module):
    def __init__(self, empirical_params=None, feature_indices=None):
        super(PINNEngine, self).__init__()
        self.tcn = DilatedTCN()
        self.empirical = empirical_params if empirical_params is not None else dict(DEFAULT_EMPIRICAL)
        self.feature_indices = feature_indices if feature_indices is not None else dict(DEFAULT_FEATURE_INDICES)

        self.IDX_P0 = self.feature_indices["P0"]
        self.IDX_P1 = self.feature_indices["P1"]
        self.IDX_T0 = self.feature_indices["T0"]
        self.IDX_T1 = self.feature_indices["T1"]

        self.raw_params = nn.ParameterDict()
        for k in self.empirical:
            if k in ('q0', 'q1'):
                init_val = float(np.log(np.exp(self.empirical[k]) - 1.0))
            else:
                init_val = 0.0
            self.raw_params[k] = nn.Parameter(torch.tensor(init_val))

    def get_bounded_physics(self):
        phys = {}
        for k, exact_val in self.empirical.items():
            if k in ('q0', 'q1'):
                phys[k] = nn.functional.softplus(self.raw_params[k])
            elif k == 'T_amb':
                phys[k] = 25.0 + (35.0 - 25.0) * torch.sigmoid(self.raw_params[k])
            else:
                # ±25% bounds
                min_bound = exact_val * 0.75
                max_bound = exact_val * 1.25
                phys[k] = min_bound + (max_bound - min_bound) * torch.sigmoid(self.raw_params[k])
        return phys

    def forward(self, x):
        return self.tcn(x)


# --- Model Interpretability ---
model_temp = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES)
print(f"\n[MODEL] ========== FULL ARCHITECTURE ==========")
print(model_temp)
total_params = sum(p.numel() for p in model_temp.parameters())
trainable_params = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)
tcn_params = sum(p.numel() for p in model_temp.tcn.parameters())
phys_params = sum(p.numel() for p in model_temp.raw_params.values())
print(f"\n[MODEL] Total Parameters     : {total_params:,}")
print(f"[MODEL] Trainable Parameters : {trainable_params:,}")
print(f"[MODEL] TCN (neural) Params  : {tcn_params:,}")
print(f"[MODEL] Physics Params       : {phys_params} ({list(DEFAULT_EMPIRICAL.keys())})")

print(f"\n[MODEL] Layer-by-layer breakdown:")
for name, param in model_temp.named_parameters():
    print(f"[MODEL]   {name:>40s} | shape={str(list(param.shape)):>15s} | numel={param.numel():>6d} | requires_grad={param.requires_grad}")

print(f"\n[MODEL] Initial physics parameter values:")
init_phys = model_temp.get_bounded_physics()
for k in DEFAULT_EMPIRICAL:
    raw = model_temp.raw_params[k].item()
    bounded = init_phys[k].item()
    if k in ('q0', 'q1'):
        print(f"[MODEL]   {k:>3s}: raw={raw:.6f} → softplus → {bounded:.6f} (init guess={DEFAULT_EMPIRICAL[k]:.6f})")
    else:
        sig = torch.sigmoid(model_temp.raw_params[k]).item()
        print(f"[MODEL]   {k:>3s}: raw={raw:.6f} → sigmoid={sig:.4f} → bounded={bounded:.6f} (empirical={DEFAULT_EMPIRICAL[k]:.6f})")
del model_temp, init_phys


# ==========================================
# 2b. PRE-BATCHED STREAMING DATA PIPELINE
# ==========================================
# FIX #3: Instead of yielding individual windows (millions of Python generator calls
# per epoch), this pipeline accumulates windows into a RAM buffer (~140 MB per worker)
# and yields complete, ready-made batch tensors. DataLoader with batch_size=None passes
# them straight through → eliminates the main CPU bottleneck.

def make_windows_prebatched(src, window_size=95, stride=10, batch_size=16384, is_train=True):
    """
    Accumulates sliding windows into a buffer, shuffles within the buffer,
    and yields pre-formed batch tensors directly.

    Memory usage: ~4 batches × 16384 × 95 × 6 × 4 bytes ≈ 140 MB per worker.
    """
    buffer = []
    buffer_count = 0
    # Accumulate 4 batches worth before shuffling & yielding
    BUFFER_TARGET = batch_size * 4

    for key, npy_array in src:
        n_rows = npy_array.shape[0]
        if n_rows < window_size:
            continue

        # Vectorized sliding window — produces all windows at once
        windows = np.lib.stride_tricks.sliding_window_view(
            npy_array, window_shape=window_size, axis=0
        )
        windows = np.swapaxes(windows, 1, 2)  # [num_windows, window_size, features]
        windows = windows[::stride]            # Apply stride

        # Append the entire chunk (NOT individual windows)
        buffer.append(windows)
        buffer_count += windows.shape[0]

        # When buffer is full, concatenate, shuffle, yield full batches
        if buffer_count >= BUFFER_TARGET:
            concat = np.concatenate(buffer, axis=0)
            if is_train:
                np.random.shuffle(concat)

            # Yield full batches
            num_full = concat.shape[0] // batch_size
            for i in range(num_full):
                batch = concat[i * batch_size : (i + 1) * batch_size]
                yield torch.from_numpy(batch.copy().astype(np.float32))

            # Keep remainder in buffer
            remainder_start = num_full * batch_size
            if remainder_start < concat.shape[0]:
                buffer = [concat[remainder_start:].copy()]
                buffer_count = concat.shape[0] - remainder_start
            else:
                buffer = []
                buffer_count = 0

    # Flush remaining windows as final batch(es)
    if buffer_count > 0:
        concat = np.concatenate(buffer, axis=0)
        if is_train:
            np.random.shuffle(concat)
        for i in range(0, concat.shape[0], batch_size):
            batch = concat[i : i + batch_size]
            yield torch.from_numpy(batch.copy().astype(np.float32))


# FIX #4: prefetch_factor=4 and persistent_workers=True for throughput
def create_dataloader(shard_list, is_train=True):
    shard_shuffle_val = 100 if is_train else 0
    dataset = (
        wds.WebDataset(shard_list, shardshuffle=shard_shuffle_val)
        .decode()
        .to_tuple("__key__", "data.npy")
    )
    # Compose the pre-batching pipeline
    dataset = dataset.compose(
        lambda src: make_windows_prebatched(
            src, window_size=WINDOW_SIZE, stride=STRIDE,
            batch_size=BATCH_SIZE, is_train=is_train
        )
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=None,        # Each item IS a full batch tensor
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=4,       # FIX: Increased prefetch (was 2)
        persistent_workers=True, # FIX: Keep workers alive between epochs
    )


train_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "train").glob("*.tar"))]
val_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "val").glob("*.tar"))]
print(f"\n[DATA] Train shards: {len(train_files)} files")
print(f"[DATA] Val shards  : {len(val_files)} files")
if train_files:
    print(f"[DATA]   First train shard: {Path(train_files[0]).name}")
    print(f"[DATA]   Last  train shard: {Path(train_files[-1]).name}")
print(f"[DATA] Pipeline: Pre-batched streaming (buffer ≈ {BATCH_SIZE * 4 * WINDOW_SIZE * 6 * 4 / 1024**2:.0f} MB/worker)")

train_loader = create_dataloader(train_files, is_train=True)
val_loader = create_dataloader(val_files, is_train=False)


# ==========================================
# 3. INTEGRATED-FORM PHYSICS LOSS (FP32)
# ==========================================
# FIX #1 (CRITICAL): Removed @torch.jit.script and _compiled_physics_rollout.
# The JIT version passed physics params via .item() (converting nn.Parameter tensors
# to Python floats), which severed the autograd graph. Physics parameters could NEVER
# receive gradients and were frozen at initialization.
#
# This version keeps all physics parameters as 0-d tensors throughout the integration
# loop, preserving the gradient graph so that C0, C1, h0, h1, k01, k10, q0, q1, T_amb
# are all fully learnable.

def compute_physics_targets_norm(model, full_data_block_abs, DT, mins_t, ranges_t):
    """
    Integrated-form physics loss in normalized [0,1] space.

    Euler-forward integrates the 2-node thermal ODE for 45 steps (from t=49 to t=93)
    using TENSOR physics parameters (gradient graph intact).

    Returns:
        T_phys_norm: [B, 2] normalized physics-predicted temperatures at t=94
        phys: dict of bounded physics parameter tensors
    """
    phys = model.get_bounded_physics()

    # Pre-extract contiguous power slices for all 45 integration steps [B, 45]
    P0_all = full_data_block_abs[:, 49:94, model.IDX_P0].float().contiguous()
    P1_all = full_data_block_abs[:, 49:94, model.IDX_P1].float().contiguous()

    # Pre-compute physics scalars as TENSORS (NOT .item() floats!)
    # This preserves the autograd graph so physics params receive gradients.
    inv_C0 = 1.0 / phys["C0"]
    inv_C1 = 1.0 / phys["C1"]
    h0 = phys["h0"]
    h1 = phys["h1"]
    k01 = phys["k01"]
    k10 = phys["k10"]
    q0 = phys["q0"]
    q1 = phys["q1"]
    T_amb = phys["T_amb"]

    # Start at the end of the history window (step 50, index 49)
    T0_curr = full_data_block_abs[:, 49, model.IDX_T0].float()
    T1_curr = full_data_block_abs[:, 49, model.IDX_T1].float()

    # Integrate forward for the next 45 steps using pre-extracted slices
    for step in range(45):
        P0 = P0_all[:, step]
        P1 = P1_all[:, step]

        T0_curr = T0_curr + DT * inv_C0 * (P0 + k01 * P1 - h0 * (T0_curr - T_amb) + q0)
        T1_curr = T1_curr + DT * inv_C1 * (P1 + k10 * P0 - h1 * (T1_curr - T_amb) + q1)

    # Normalize final integrated step (t+45 = index 94 = last step of window)
    T0_phys_norm = (T0_curr - mins_t[4]) / ranges_t[4]
    T1_phys_norm = (T1_curr - mins_t[5]) / ranges_t[5]

    return torch.stack([T0_phys_norm, T1_phys_norm], dim=1), phys


# ==========================================
# 4. ReLoBRaLo ADAPTIVE LAMBDA
# ==========================================
class ReLoBRaLo:
    """
    Relative Loss Balancing with Random Lookback (ReLoBRaLo)
    Based on: Bischof & Kraus, 2021.
    """
    def __init__(self, num_losses=2, alpha=0.999, temperature=1.0, rho=0.999):
        self.num_losses = num_losses
        self.alpha = alpha
        self.temperature = temperature
        self.rho = rho
        self.initial_losses = None
        self.previous_losses = None
        self.weights = [1.0] * num_losses

    def get_weights(self, current_losses):
        assert len(current_losses) == self.num_losses
        if self.initial_losses is None:
            self.initial_losses = list(current_losses)
            self.previous_losses = list(current_losses)
            return list(self.weights)

        use_initial = random.random() < self.rho
        reference = self.initial_losses if use_initial else self.previous_losses

        ratios = []
        for i in range(self.num_losses):
            ref_val = reference[i] if reference[i] > 1e-12 else 1e-12
            ratios.append(current_losses[i] / ref_val)

        max_ratio = max(ratios)
        exp_ratios = [np.exp((r - max_ratio) / self.temperature) for r in ratios]
        sum_exp = sum(exp_ratios)
        instant_weights = [(self.num_losses * e / sum_exp) for e in exp_ratios]

        self.weights = [
            self.alpha * w_old + (1.0 - self.alpha) * w_new
            for w_old, w_new in zip(self.weights, instant_weights)
        ]
        self.previous_losses = list(current_losses)
        return list(self.weights)

    def state_dict(self):
        return {
            'initial_losses': self.initial_losses, 'previous_losses': self.previous_losses,
            'weights': self.weights, 'alpha': self.alpha,
            'temperature': self.temperature, 'rho': self.rho,
        }

    def load_state_dict(self, state):
        self.initial_losses = state['initial_losses']
        self.previous_losses = state['previous_losses']
        self.weights = state['weights']
        self.alpha = state['alpha']
        self.temperature = state['temperature']
        self.rho = state['rho']


# ==========================================
# 5. EPOCH 1 WALKTHROUGH (extracted)
# ==========================================
def epoch1_walkthrough(model, relobralo, current_phys, DT,
                       num_train_batches, total_train_loss, total_train_loss_data, total_train_loss_phys,
                       avg_train_loss, avg_train_data, avg_train_phys,
                       num_val_batches, total_val_loss, total_val_loss_data, total_val_loss_phys,
                       avg_val_loss, avg_val_data, avg_val_phys,
                       lambda_data, lambda_phys,
                       Y_val_pred_norm, Y_val_true_norm, T_phys_norm_v,
                       X_val, loss_data_v, loss_phys_v,
                       mins_cpu, ranges_cpu):
    """Complete mathematical walkthrough printed after Epoch 1 for verification."""
    SEP = "=" * 90
    print(f"\n[MATH] {SEP}")
    print(f"[MATH] >>> EPOCH 1: COMPLETE MATHEMATICAL WALKTHROUGH & PROOF <<<")
    print(f"[MATH] {SEP}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 1: TRAINING LOSS AGGREGATION ---")
    print(f"[MATH] Total train batches processed: {num_train_batches}")
    print(f"[MATH]   total_train_loss      = {total_train_loss:.8f}")
    print(f"[MATH]   total_train_loss_data  = {total_train_loss_data:.8f}")
    print(f"[MATH]   total_train_loss_phys  = {total_train_loss_phys:.8f}")
    print(f"[MATH] Averaging:")
    print(f"[MATH]   avg_train_loss = {total_train_loss:.8f} / {num_train_batches} = {avg_train_loss:.8f}")
    print(f"[MATH]   avg_train_data = {total_train_loss_data:.8f} / {num_train_batches} = {avg_train_data:.8f}")
    print(f"[MATH]   avg_train_phys = {total_train_loss_phys:.8f} / {num_train_batches} = {avg_train_phys:.8f}")
    print(f"[MATH]   NOTE: Physics loss uses INTEGRATED FORM in normalized [0,1] space.")
    print(f"[MATH]         Both losses are MSE in [0,1] → inherently comparable.")
    print(f"[MATH]   Final lambdas: λ_data={lambda_data:.8f}, λ_phys={lambda_phys:.8e}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 2: VALIDATION LOSS ---")
    print(f"[MATH] Val batches: {num_val_batches}, λ_data={lambda_data:.8f}, λ_phys={lambda_phys:.8e}")
    print(f"[MATH]   avg_val_loss = {avg_val_loss:.8f}")
    print(f"[MATH]   avg_val_data = {avg_val_data:.8f}")
    print(f"[MATH]   avg_val_phys = {avg_val_phys:.8f}")
    reconstructed_val = lambda_data * avg_val_data + lambda_phys * avg_val_phys
    match_str = "✓ MATCH" if abs(reconstructed_val - avg_val_loss) < 1e-6 else "✗ MISMATCH"
    print(f"[MATH]   Reconstruction: λd*data + λp*phys = {reconstructed_val:.8f} vs {avg_val_loss:.8f} → {match_str}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 3: DATA LOSS (last val batch, Row 0) ---")
    print(f"[MATH]   Y_pred_norm[0] = [{Y_val_pred_norm[0, 0].item():.8f}, {Y_val_pred_norm[0, 1].item():.8f}]")
    print(f"[MATH]   Y_true_norm[0] = [{Y_val_true_norm[0, 0].item():.8f}, {Y_val_true_norm[0, 1].item():.8f}]")
    d0 = Y_val_pred_norm[0, 0].item() - Y_val_true_norm[0, 0].item()
    d1 = Y_val_pred_norm[0, 1].item() - Y_val_true_norm[0, 1].item()
    print(f"[MATH]   Diffs: T0={d0:.8f}, T1={d1:.8f}")
    print(f"[MATH]   Squared: T0²={d0**2:.8f}, T1²={d1**2:.8f}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 4: INTEGRATED-FORM PHYSICS LOSS (Row 0) ---")
    print(f"[MATH]   FORMULA: T_phys = T_current + DT * (1/C) * (P + k*P' - h*(T-Tamb) + q)")
    print(f"[MATH]            Integrated over 45 steps from t=49 to t=93")
    print(f"[MATH]            T_phys_norm = (T_phys - min) / range")
    print(f"[MATH]            loss_phys = MSE(Y_pred_norm, T_phys_norm)")
    X_val_last = X_val[:, -1, :]
    p0 = X_val_last[0, model.IDX_P0].item()
    p1 = X_val_last[0, model.IDX_P1].item()
    t0_t = X_val_last[0, model.IDX_T0].item()
    t1_t = X_val_last[0, model.IDX_T1].item()

    C0, C1 = current_phys['C0'].item(), current_phys['C1'].item()
    h0, h1 = current_phys['h0'].item(), current_phys['h1'].item()
    k01, k10 = current_phys['k01'].item(), current_phys['k10'].item()
    q0, q1 = current_phys['q0'].item(), current_phys['q1'].item()
    t_amb = current_phys['T_amb'].item()

    print(f"[MATH]   Inputs (last step): P0={p0:.4f}W, P1={p1:.4f}W, T0_t={t0_t:.4f}°C, T1_t={t1_t:.4f}°C, T_amb={t_amb:.4f}°C")

    print(f"[MATH]   Physics params:")
    for k in model.empirical:
        pct = ((current_phys[k].item() / model.empirical[k]) - 1) * 100
        ptype = "softplus" if k in ('q0', 'q1') else "sigmoid"
        print(f"[MATH]     {k:>3s} = {current_phys[k].item():.6f} (emp={model.empirical[k]:.6f}, Δ={pct:+.2f}%, {ptype})")

    print(f"[MATH]   T_phys_norm (after 45-step integration):")
    print(f"[MATH]         Tensor T_phys_norm[0,0] = {T_phys_norm_v[0, 0].item():.8f}")
    print(f"[MATH]         Y_pred_norm[0,0]        = {Y_val_pred_norm[0, 0].item():.8f}")
    dp0 = Y_val_pred_norm[0, 0].item() - T_phys_norm_v[0, 0].item()
    print(f"[MATH]         Physics diff: {dp0:.8f}, squared: {dp0**2:.8f}")

    print(f"[MATH]         Tensor T_phys_norm[0,1] = {T_phys_norm_v[0, 1].item():.8f}")
    print(f"[MATH]         Y_pred_norm[0,1]        = {Y_val_pred_norm[0, 1].item():.8f}")
    dp1 = Y_val_pred_norm[0, 1].item() - T_phys_norm_v[0, 1].item()
    print(f"[MATH]         Physics diff: {dp1:.8f}, squared: {dp1**2:.8f}")

    print(f"[MATH]")
    print(f"[MATH]   Last batch loss_data = {loss_data_v.item():.8f}")
    print(f"[MATH]   Last batch loss_phys = {loss_phys_v.item():.8f}")
    print(f"[MATH]   RATIO: loss_phys / loss_data = {loss_phys_v.item() / max(loss_data_v.item(), 1e-12):.2f}x")
    print(f"[MATH]   (Should be within ~0.1x to ~10x for effective balancing)")

    print(f"[MATH]")
    print(f"[MATH] --- PART 5: ReLoBRaLo ---")
    print(f"[MATH]   Initial: data={relobralo.initial_losses[0]:.8f}, phys={relobralo.initial_losses[1]:.8f}")
    print(f"[MATH]   Final:   λ_data={relobralo.weights[0]:.8f}, λ_phys={relobralo.weights[1]:.8f}")
    print(f"[MATH]   Sum = {relobralo.weights[0] + relobralo.weights[1]:.6f} (should ≈ 2.0)")

    print(f"[MATH] {SEP}\n")


# ==========================================
# 6. TRAINING INFRASTRUCTURE & AUTO-RESUME
# ==========================================
EPOCHS = 50
LEARNING_RATE = 1e-3

print(f"\n[CONFIG] EPOCHS={EPOCHS}, LEARNING_RATE={LEARNING_RATE}, DT={DT}")
print(f"[CONFIG] Physics loss: INTEGRATED FORM in normalized [0,1] space")

model = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6)
mse_loss = nn.MSELoss()
scaler = torch.amp.GradScaler('cuda')

relobralo = ReLoBRaLo(num_losses=2, alpha=0.999, temperature=1.0, rho=0.999)
print(f"[CONFIG] ReLoBRaLo: alpha={relobralo.alpha}, temperature={relobralo.temperature}, rho={relobralo.rho}")
print(f"[CONFIG] Optimizer: Adam (lr={LEARNING_RATE})")
print(f"[CONFIG] Scheduler: ReduceLROnPlateau (factor=0.5, patience=4, min_lr=1e-6)")
print(f"[CONFIG] GradScaler: enabled (mixed precision FP16/FP32)")
print(f"[CONFIG] Gradient Clipping: max_norm=1.0")

start_epoch = 1
best_val_loss = float('inf')
RESUME_PATH = SAVE_DIR / f"{VERSION_PREFIX}_resume_checkpoint.pt"
BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
gpu_log_path = SAVE_DIR / f"{VERSION_PREFIX}_gpu_stats.csv"
epoch_log_path = SAVE_DIR / f"{VERSION_PREFIX}_epoch_logs.csv"

if RESUME_PATH.exists():
    print(f"\n[SYSTEM] Found existing checkpoint at: {str(RESUME_PATH).replace(prefix, '')}")
    print(f"[SYSTEM] Resuming training...")
    checkpoint = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    if 'relobralo_state' in checkpoint:
        relobralo.load_state_dict(checkpoint['relobralo_state'])
        print(f"[SYSTEM] ReLoBRaLo restored: λ_data={relobralo.weights[0]:.6f}, λ_phys={relobralo.weights[1]:.6f}")
    torch.set_rng_state(checkpoint['torch_rng'].cpu().to(torch.uint8) if isinstance(checkpoint['torch_rng'], torch.Tensor) else torch.ByteTensor(checkpoint['torch_rng']))
    np.random.set_state(checkpoint['numpy_rng'])
    if torch.cuda.is_available() and 'cuda_rng' in checkpoint and checkpoint['cuda_rng'] is not None:
        cuda_rng = checkpoint['cuda_rng']
        if isinstance(cuda_rng, torch.Tensor):
            cuda_rng = cuda_rng.cpu().to(torch.uint8)
        else:
            cuda_rng = torch.ByteTensor(cuda_rng)
        torch.cuda.set_rng_state(cuda_rng)
    print(f"[SYSTEM] Resumed from Epoch {checkpoint['epoch']}. Best Val Loss: {best_val_loss:.6f}")
else:
    print(f"\n[SYSTEM] No checkpoint found. Starting fresh training.")

gpu_log_file = open(gpu_log_path, "w")
gpu_logger = subprocess.Popen(
    ["nvidia-smi", "-i", str(GPU_ID), "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu",
     "--format=csv", "-l", "5"],
    stdout=gpu_log_file, stderr=subprocess.STDOUT
)

csv_mode = 'a' if RESUME_PATH.exists() and epoch_log_path.exists() else 'w'
epoch_log_file = open(epoch_log_path, mode=csv_mode, newline='')
epoch_writer = csv.writer(epoch_log_file)
if csv_mode == 'w':
    epoch_writer.writerow([
        'Epoch', 'Train_Loss_Total', 'Train_Loss_Data', 'Train_Loss_Phys',
        'Val_Loss_Total', 'Val_Loss_Data', 'Val_Loss_Phys',
        'Lambda_Data', 'Lambda_Phys',
        'C0', 'C1', 'h0', 'h1', 'k01', 'k10', 'q0', 'q1', 'T_amb'
    ])

# ==========================================
# 7. PRE-TRAINING DATA INSPECTION
# ==========================================
# FIX #9: Pre-batched pipeline yields tensors directly (no keys, no tuple wrapping)
print(f"\n[INSPECT] ========== FIRST BATCH DEEP INSPECTION ==========")
_inspect_block = next(iter(train_loader))
print(f"[INSPECT] raw data_block  : shape={list(_inspect_block.shape)}, dtype={_inspect_block.dtype}")
print(f"[INSPECT]   Interpretation: ({_inspect_block.shape[0]} samples) x ({_inspect_block.shape[1]} timesteps) x ({_inspect_block.shape[2]} features)")
print(f"[INSPECT]   Features: {scalars['feature_order']}")
print(f"[INSPECT] First sample, first 3 timesteps (RAW ABSOLUTE):")
for _t in range(min(3, _inspect_block.shape[1])):
    _row = _inspect_block[0, _t, :].numpy()
    print(f"[INSPECT]   t={_t}:")
    print(f"[INSPECT]       Power : GPU0={_row[0]:>7.4f}W | GPU1={_row[1]:>7.4f}W")
    print(f"[INSPECT]       Util  : GPU0={_row[2]:>7.4f}% | GPU1={_row[3]:>7.4f}%")
    print(f"[INSPECT]       Temp  : GPU0={_row[4]:>7.4f}C | GPU1={_row[5]:>7.4f}C")

_X_seq = _inspect_block[:, :50, :]
_Y_true_abs = _inspect_block[:, -1, 4:6]
_Y_true_norm = (_Y_true_abs - mins[4:6].cpu()) / ranges[4:6].cpu()
_X_norm = (_X_seq - mins.cpu()) / ranges.cpu()

print(f"\n[INSPECT] After slicing & normalization:")
print(f"[INSPECT]   X_seq (input)      : shape={list(_X_seq.shape)}, dtype={_X_seq.dtype}")
print(f"[INSPECT]   Y_true_abs (target): shape={list(_Y_true_abs.shape)}, dtype={_Y_true_abs.dtype}")
print(f"[INSPECT]   X_norm (normalized): shape={list(_X_norm.shape)}, dtype={_X_norm.dtype}")
print(f"[INSPECT]   Y_true_norm        : shape={list(_Y_true_norm.shape)}, dtype={_Y_true_norm.dtype}")
print(f"[INSPECT] Sample Row 0:")
print(f"[INSPECT]   Absolute   : T0={_Y_true_abs[0, 0].item():.4f}, T1={_Y_true_abs[0, 1].item():.4f}")
print(f"[INSPECT]   Normalized : T0={_Y_true_norm[0, 0].item():.6f}, T1={_Y_true_norm[0, 1].item():.6f}")
print(f"[INSPECT] ========== END FIRST BATCH INSPECTION ==========\n")

del _inspect_block, _X_seq, _Y_true_abs, _Y_true_norm, _X_norm, _row

print(f"\n{'='*80}")
print(f"[TRAINING] INITIATING HPC MODEL TRAINING [{VERSION_PREFIX}]")
print(f"[TRAINING] Using ReLoBRaLo adaptive loss balancing")
print(f"[TRAINING] Physics loss: INTEGRATED FORM in normalized [0,1] space")
print(f"[TRAINING] Data pipeline: PRE-BATCHED streaming (zero per-window overhead)")
print(f"[TRAINING] Epochs: {start_epoch} → {EPOCHS} | Device: {device}")
print(f"{'='*80}")

# ==========================================
# 8. MAIN EPOCH LOOP
# ==========================================
try:
    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start_time = time.time()
        model.train()

        total_train_loss = 0.0
        total_train_loss_data = 0.0
        total_train_loss_phys = 0.0
        num_train_batches = 0

        # FIX #8: Removed hardcoded total= (batch count changes with pre-batched pipeline)
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [Train]", leave=False)

        # FIX #9: No tuple unpacking — pre-batched pipeline yields tensors directly
        for batch_idx, data_block in enumerate(train_pbar):
            # FIX #6: Transfer entire block once with non_blocking=True, then slice on GPU
            data_block = data_block.to(device, non_blocking=True)

            X_seq = data_block[:, :50, :]       # First 50 steps for TCN input
            Y_true_abs = data_block[:, -1, 4:6]  # Target is the 95th step (index 94)
            Y_true_norm = (Y_true_abs - mins[4:6]) / ranges[4:6]
            X_norm = (X_seq - mins) / ranges

            # FIX #7: set_to_none=True avoids memset, saves time
            optimizer.zero_grad(set_to_none=True)

            # --- Forward pass (FP16 for data loss) ---
            with torch.amp.autocast('cuda'):
                Y_pred_norm = model(X_norm)
                loss_data = mse_loss(Y_pred_norm, Y_true_norm)

            # --- Physics loss: integrated form in normalized space (FP32) ---
            T_phys_norm, _ = compute_physics_targets_norm(model, data_block, DT, mins, ranges)
            loss_phys = mse_loss(Y_pred_norm.float(), T_phys_norm)

            # --- ReLoBRaLo: get adaptive weights ---
            lambdas = relobralo.get_weights([loss_data.item(), loss_phys.item()])
            lambda_data = lambdas[0]
            lambda_phys = lambdas[1]

            # --- Composite loss ---
            loss_total = lambda_data * loss_data + lambda_phys * loss_phys

            scaler.scale(loss_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            # FIX #10: Verify gradient flow on first batch of first epoch
            if batch_idx == 0 and epoch == start_epoch:
                print(f"\n[VERIFY] ========== PHYSICS PARAMETER GRADIENT CHECK ==========")
                for k, p in model.raw_params.items():
                    if p.grad is not None:
                        print(f"[VERIFY]   {k:>5s}.grad = {p.grad.item():+.8e}  ✓ RECEIVING GRADIENTS")
                    else:
                        print(f"[VERIFY]   {k:>5s}.grad = None  ✗ NO GRADIENT (BUG!)")
                print(f"[VERIFY] ==========================================================\n")

            total_train_loss += loss_total.item()
            total_train_loss_data += loss_data.item()
            total_train_loss_phys += loss_phys.item()
            num_train_batches += 1

            train_pbar.set_postfix({
                "Loss": f"{loss_total.item():.4f}",
                "Ld": f"{loss_data.item():.2e}",
                "Lp": f"{loss_phys.item():.2e}",
                "λd": f"{lambda_data:.3f}",
                "λp": f"{lambda_phys:.3f}"
            })

        avg_train_loss = total_train_loss / num_train_batches
        avg_train_data = total_train_loss_data / num_train_batches
        avg_train_phys = total_train_loss_phys / num_train_batches

        # --- VALIDATION LOOP ---
        model.eval()
        total_val_loss = 0.0
        total_val_loss_data = 0.0
        total_val_loss_phys = 0.0
        num_val_batches = 0

        val_pbar = tqdm(val_loader, desc="Validation", leave=False)

        with torch.no_grad():
            for batch_idx, val_block in enumerate(val_pbar):
                val_block = val_block.to(device, non_blocking=True)

                X_val = val_block[:, :50, :]
                Y_val_true_abs = val_block[:, -1, 4:6]
                Y_val_true_norm = (Y_val_true_abs - mins[4:6]) / ranges[4:6]
                X_val_norm = (X_val - mins) / ranges

                with torch.amp.autocast('cuda'):
                    Y_val_pred_norm = model(X_val_norm)
                    loss_data_v = mse_loss(Y_val_pred_norm, Y_val_true_norm)

                T_phys_norm_v, _ = compute_physics_targets_norm(model, val_block, DT, mins, ranges)
                loss_phys_v = mse_loss(Y_val_pred_norm.float(), T_phys_norm_v)

                loss_data_v_scalar = loss_data_v.item()
                loss_phys_v_scalar = loss_phys_v.item()
                loss_total_v = lambda_data * loss_data_v_scalar + lambda_phys * loss_phys_v_scalar

                total_val_loss += loss_total_v
                total_val_loss_data += loss_data_v_scalar
                total_val_loss_phys += loss_phys_v_scalar
                num_val_batches += 1

        avg_val_loss = total_val_loss / num_val_batches
        avg_val_data = total_val_loss_data / num_val_batches
        avg_val_phys = total_val_loss_phys / num_val_batches
        epoch_duration = time.time() - epoch_start_time

        current_phys = model.get_bounded_physics()
        epoch_writer.writerow([
            epoch, avg_train_loss, avg_train_data, avg_train_phys,
            avg_val_loss, avg_val_data, avg_val_phys,
            lambda_data, lambda_phys,
            *[current_phys[k].item() for k in ['C0','C1','h0','h1','k01','k10','q0','q1','T_amb']]
        ])
        epoch_log_file.flush()

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        lr_flag = f" [LR Reduced to {new_lr:.2e}]" if new_lr < old_lr else ""
        best_flag = ""

        checkpoint_state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_loss': best_val_loss,
            'relobralo_state': relobralo.state_dict(),
            'torch_rng': torch.get_rng_state(),
            'numpy_rng': np.random.get_state(),
            'cuda_rng': torch.cuda.get_rng_state() if torch.cuda.is_available() else None
        }
        torch.save(checkpoint_state, RESUME_PATH)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            checkpoint_state['best_val_loss'] = best_val_loss
            torch.save(checkpoint_state, BEST_PATH)
            best_flag = " [*]"

        if epoch == 1:
            epoch1_walkthrough(
                model=model, relobralo=relobralo, current_phys=current_phys, DT=DT,
                num_train_batches=num_train_batches,
                total_train_loss=total_train_loss, total_train_loss_data=total_train_loss_data,
                total_train_loss_phys=total_train_loss_phys,
                avg_train_loss=avg_train_loss, avg_train_data=avg_train_data, avg_train_phys=avg_train_phys,
                num_val_batches=num_val_batches,
                total_val_loss=total_val_loss, total_val_loss_data=total_val_loss_data,
                total_val_loss_phys=total_val_loss_phys,
                avg_val_loss=avg_val_loss, avg_val_data=avg_val_data, avg_val_phys=avg_val_phys,
                lambda_data=lambda_data, lambda_phys=lambda_phys,
                Y_val_pred_norm=Y_val_pred_norm, Y_val_true_norm=Y_val_true_norm,
                T_phys_norm_v=T_phys_norm_v,
                X_val=X_val, loss_data_v=loss_data_v, loss_phys_v=loss_phys_v,
                mins_cpu=mins.cpu(), ranges_cpu=ranges.cpu()
            )

        print(f"[TRAINING] Epoch {epoch:02d}/{EPOCHS} | Time: {epoch_duration:.1f}s | "
              f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | "
              f"λd:{lambda_data:.3f} λp:{lambda_phys:.3f}{best_flag}{lr_flag}")

except KeyboardInterrupt:
    print("\n[!] Training manually interrupted. Checkpoints are safe.")
finally:
    try:
        gpu_logger.kill()
    except ProcessLookupError:
        pass
    gpu_log_file.close()
    epoch_log_file.close()
    print(f"[SYSTEM] Background GPU logging terminated. Epoch logs securely saved.")



[SYSTEM] Live GPU Status:
Fri Mar 27 18:06:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 30%   51C    P2             62W /  200W |     191MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  0


[SYSTEM] Manually locking script to GPU 0.
[SYSTEM] PyTorch initialized. Active Device: NVIDIA RTX A4500 (cuda:0)
[SYSTEM] CUDA Version: 12.6 | PyTorch Version: 2.9.1+cu126
[SYSTEM] GPU Memory: 19.6 GB
[SYSTEM] All RNG sources seeded with MASTER_SEED=42
[SYSTEM] cudnn.deterministic=True, cudnn.benchmark=True



Enter version prefix (e.g., tcn_v1) [Press Enter for default 'unknown_version']:  TCN_ReLoBRaLo_v3



[PATHS] PROJECT_DIR     : Capstone/Implementation
[PATHS] LOCAL_SHARDS_DIR: Capstone/Implementation/data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards
[PATHS] SAVE_DIR        : Capstone/Implementation/models/TCN_ReLoBRaLo_v3
[PATHS] SCALARS_PATH    : Capstone/Implementation/data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards/normalization_scalars.json

[DATA] Normalization scalars loaded successfully.
[DATA] Feature Order: ['power_draw_gpu_0_W', 'power_draw_gpu_1_W', 'utilization_gpu_0_pct', 'utilization_gpu_1_pct', 'temperature_gpu_0', 'temperature_gpu_1']
[DATA] Number of Features: 6
[DATA]   [0]             power_draw_gpu_0_W | min=     21.8100 | max=    331.0800
[DATA]   [1]             power_draw_gpu_1_W | min=     22.7800 | max=    328.8600
[DATA]   [2]          utilization_gpu_0_pct | min=      0.0000 | max=    100.0000
[DATA]   [3]          utilization_gpu_1_pct | min=      0.0000 | max=    100.0000

Epoch 01/50 [Train]: 1it [00:07,  7.81s/it, Loss=0.3961, Ld=1.92e-01, Lp=2.05e-01, λd=1.000, λp=1.000]


[VERIFY] ========== PHYSICS PARAMETER GRADIENT CHECK ==========
[VERIFY]      C0.grad = -1.68889710e-05  ✓ RECEIVING GRADIENTS
[VERIFY]      C1.grad = -9.87541425e-05  ✓ RECEIVING GRADIENTS
[VERIFY]      h0.grad = +3.32024069e-06  ✓ RECEIVING GRADIENTS
[VERIFY]      h1.grad = -1.10831294e-04  ✓ RECEIVING GRADIENTS
[VERIFY]     k01.grad = +3.10503719e-06  ✓ RECEIVING GRADIENTS
[VERIFY]     k10.grad = +1.93080677e-06  ✓ RECEIVING GRADIENTS
[VERIFY]      q0.grad = +1.97066538e-06  ✓ RECEIVING GRADIENTS
[VERIFY]      q1.grad = +1.28842712e-05  ✓ RECEIVING GRADIENTS
[VERIFY]   T_amb.grad = +1.97521382e-04  ✓ RECEIVING GRADIENTS
[VERIFY] ==========================================================




[MATH] ==========================================================================================
[MATH] >>> EPOCH 1: COMPLETE MATHEMATICAL WALKTHROUGH & PROOF <<<
[MATH] ==========================================================================================
[MATH]
[MATH] --- PART 1: TRAINING LOSS AGGREGATION ---
[MATH] Total train batches processed: 10410
[MATH]   total_train_loss      = 95.54236998
[MATH]   total_train_loss_data  = 49.17249770
[MATH]   total_train_loss_phys  = 46.36604988
[MATH] Averaging:
[MATH]   avg_train_loss = 95.54236998 / 10410 = 0.00917794
[MATH]   avg_train_data = 49.17249770 / 10410 = 0.00472358
[MATH]   avg_train_phys = 46.36604988 / 10410 = 0.00445399
[MATH]   NOTE: Physics loss uses INTEGRATED FORM in normalized [0,1] space.
[MATH]         Both losses are MSE in [0,1] → inherently comparable.
[MATH]   Final lambdas: λ_data=1.00061142, λ_phys=9.99388579e-01
[MATH]
[MATH] --- PART 2: VALIDATION LOSS ---
[MATH] Val batches: 1201, λ_data=1.00061142, λ_ph

[TRAINING] Epoch 02/50 | Time: 2257.9s | Train Loss: 0.001345 | Val Loss: 0.002890 | λd:1.001 λp:0.999



[!] Training manually interrupted. Checkpoints are safe.
[SYSTEM] Background GPU logging terminated. Epoch logs securely saved.


# Testing

In [4]:
import os
import sys
import subprocess
import torch
import torch.nn as nn
import numpy as np
import webdataset as wds
import json
import time
import csv
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. HARDWARE & PATH INITIALIZATION
# ==========================================
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt (in that order).
    Usage: python test_model.py --gpu 1   OR   GPU_ID=1 python test_model.py
    """
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

GPU_ID = select_gpu()
torch.cuda.set_device(GPU_ID)
device = torch.device(f"cuda:{GPU_ID}")

if torch.cuda.is_available():
    print(f"[SYSTEM] PyTorch initialized. Active Device: {torch.cuda.get_device_name(GPU_ID)} (cuda:{GPU_ID})")
    print(f"[SYSTEM] CUDA Version: {torch.version.cuda} | PyTorch Version: {torch.__version__}")
    print(f"[SYSTEM] GPU Memory: {torch.cuda.get_device_properties(GPU_ID).total_memory / 1024**3:.1f} GB")
else:
    print(f"[WARNING] PyTorch could not find a CUDA device. Falling back to CPU.")

# --- Paths ---
user_prefix = input("\nEnter version prefix to TEST (e.g., TCN_ReLoBRaLo_v3) [Press Enter for default 'unknown_version']: ").strip()
VERSION_PREFIX = user_prefix if user_prefix else "unknown_version"

PROJECT_DIR = Path(os.path.expanduser("~/Capstone/Implementation"))
LOCAL_SHARDS_DIR = PROJECT_DIR / "data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards"
SAVE_DIR = PROJECT_DIR / f"models/{VERSION_PREFIX}"
SCALARS_PATH = LOCAL_SHARDS_DIR / "normalization_scalars.json"
BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"

prefix = "/home/sanke24839/"

print(f"\n[PATHS] PROJECT_DIR     : {str(PROJECT_DIR).replace(prefix, '')}")
print(f"[PATHS] LOCAL_SHARDS_DIR: {str(LOCAL_SHARDS_DIR).replace(prefix, '')}")
print(f"[PATHS] SAVE_DIR        : {str(SAVE_DIR).replace(prefix, '')}")
print(f"[PATHS] BEST_PATH       : {str(BEST_PATH).replace(prefix, '')}")

if not BEST_PATH.exists():
    print(f"\n[FATAL] Best model checkpoint NOT FOUND at: {BEST_PATH}")
    print(f"[FATAL] Make sure training has completed and produced a best model.")
    sys.exit(1)

with open(SCALARS_PATH, 'r') as f:
    scalars = json.load(f)

print(f"\n[DATA] Normalization scalars loaded successfully.")
print(f"[DATA] Feature Order: {scalars['feature_order']}")
print(f"[DATA] Number of Features: {len(scalars['feature_order'])}")

for i, feat in enumerate(scalars['feature_order']):
    print(f"[DATA]   [{i}] {feat:>30s} | min={scalars['global_minimums'][feat]:>12.4f} | max={scalars['global_maximums'][feat]:>12.4f}")

mins = torch.tensor([scalars['global_minimums'][c] for c in scalars['feature_order']], device=device)
maxs = torch.tensor([scalars['global_maximums'][c] for c in scalars['feature_order']], device=device)
ranges = maxs - mins
ranges[ranges == 0] = 1.0

# ==========================================
# 2. ARCHITECTURE (must match training exactly)
# ==========================================
DEFAULT_EMPIRICAL = {
    "C0": 143.22, "C1": 140.25, "h0": 4.8713, "h1": 5.3871,
    "k01": 0.078038, "k10": 0.028120, "q0": 35.0, "q1": 35.0,
    "T_amb": 30.5
}

DEFAULT_FEATURE_INDICES = {
    "P0": 0, "P1": 1, "T0": 4, "T1": 5
}

BATCH_SIZE = 16384
NUM_WORKERS = 8
WINDOW_SIZE = 95
STRIDE = 10
DT = 0.11


class DilatedTCN(nn.Module):
    def __init__(self, num_inputs=6, num_channels=[32, 64, 128, 256, 512], kernel_size=3):
        super(DilatedTCN, self).__init__()
        layers = []
        in_channels = num_inputs
        for i, out_channels in enumerate(num_channels):
            dilation_size = 2 ** i
            padding = (kernel_size - 1) * dilation_size
            layers += [
                nn.ConstantPad1d((padding, 0), 0.0),
                nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
            ]
            in_channels = out_channels
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.tcn(x)
        return self.fc(out[:, :, -1])


class PINNEngine(nn.Module):
    def __init__(self, empirical_params=None, feature_indices=None):
        super(PINNEngine, self).__init__()
        self.tcn = DilatedTCN()
        self.empirical = empirical_params if empirical_params is not None else dict(DEFAULT_EMPIRICAL)
        self.feature_indices = feature_indices if feature_indices is not None else dict(DEFAULT_FEATURE_INDICES)

        self.IDX_P0 = self.feature_indices["P0"]
        self.IDX_P1 = self.feature_indices["P1"]
        self.IDX_T0 = self.feature_indices["T0"]
        self.IDX_T1 = self.feature_indices["T1"]

        self.raw_params = nn.ParameterDict()
        for k in self.empirical:
            if k in ('q0', 'q1'):
                init_val = float(np.log(np.exp(self.empirical[k]) - 1.0))
            else:
                init_val = 0.0
            self.raw_params[k] = nn.Parameter(torch.tensor(init_val))

    def get_bounded_physics(self):
        phys = {}
        for k, exact_val in self.empirical.items():
            if k in ('q0', 'q1'):
                phys[k] = nn.functional.softplus(self.raw_params[k])
            elif k == 'T_amb':
                phys[k] = 25.0 + (35.0 - 25.0) * torch.sigmoid(self.raw_params[k])
            else:
                min_bound = exact_val * 0.75
                max_bound = exact_val * 1.25
                phys[k] = min_bound + (max_bound - min_bound) * torch.sigmoid(self.raw_params[k])
        return phys

    def forward(self, x):
        return self.tcn(x)


# ==========================================
# 3. DATA PIPELINE (same pre-batched streaming)
# ==========================================
def make_windows_prebatched(src, window_size=95, stride=10, batch_size=16384, is_train=False):
    """
    Pre-batched streaming for test evaluation (no shuffle).
    """
    buffer = []
    buffer_count = 0
    BUFFER_TARGET = batch_size * 4

    for key, npy_array in src:
        n_rows = npy_array.shape[0]
        if n_rows < window_size:
            continue

        windows = np.lib.stride_tricks.sliding_window_view(
            npy_array, window_shape=window_size, axis=0
        )
        windows = np.swapaxes(windows, 1, 2)
        windows = windows[::stride]

        buffer.append(windows)
        buffer_count += windows.shape[0]

        if buffer_count >= BUFFER_TARGET:
            concat = np.concatenate(buffer, axis=0)

            num_full = concat.shape[0] // batch_size
            for i in range(num_full):
                batch = concat[i * batch_size : (i + 1) * batch_size]
                yield torch.from_numpy(batch.copy().astype(np.float32))

            remainder_start = num_full * batch_size
            if remainder_start < concat.shape[0]:
                buffer = [concat[remainder_start:].copy()]
                buffer_count = concat.shape[0] - remainder_start
            else:
                buffer = []
                buffer_count = 0

    if buffer_count > 0:
        concat = np.concatenate(buffer, axis=0)
        for i in range(0, concat.shape[0], batch_size):
            batch = concat[i : i + batch_size]
            yield torch.from_numpy(batch.copy().astype(np.float32))


def create_test_dataloader(shard_list):
    dataset = (
        wds.WebDataset(shard_list, shardshuffle=0)
        .decode()
        .to_tuple("__key__", "data.npy")
    )
    dataset = dataset.compose(
        lambda src: make_windows_prebatched(
            src, window_size=WINDOW_SIZE, stride=STRIDE,
            batch_size=BATCH_SIZE, is_train=False
        )
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=None,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=4,
        persistent_workers=False,  # Not needed for single-pass test
    )


# ==========================================
# 4. PHYSICS INTEGRATION (same as training)
# ==========================================
def compute_physics_targets_norm(model, full_data_block_abs, DT, mins_t, ranges_t):
    """
    Integrated-form physics: Euler-forward 45 steps from t=49 to t=93.
    Returns [B, 2] normalized physics-predicted temperatures at t=94.
    """
    phys = model.get_bounded_physics()

    P0_all = full_data_block_abs[:, 49:94, model.IDX_P0].float().contiguous()
    P1_all = full_data_block_abs[:, 49:94, model.IDX_P1].float().contiguous()

    inv_C0 = 1.0 / phys["C0"]
    inv_C1 = 1.0 / phys["C1"]
    h0, h1 = phys["h0"], phys["h1"]
    k01, k10 = phys["k01"], phys["k10"]
    q0, q1 = phys["q0"], phys["q1"]
    T_amb = phys["T_amb"]

    T0_curr = full_data_block_abs[:, 49, model.IDX_T0].float()
    T1_curr = full_data_block_abs[:, 49, model.IDX_T1].float()

    for step in range(45):
        P0 = P0_all[:, step]
        P1 = P1_all[:, step]

        T0_curr = T0_curr + DT * inv_C0 * (P0 + k01 * P1 - h0 * (T0_curr - T_amb) + q0)
        T1_curr = T1_curr + DT * inv_C1 * (P1 + k10 * P0 - h1 * (T1_curr - T_amb) + q1)

    T0_phys_norm = (T0_curr - mins_t[4]) / ranges_t[4]
    T1_phys_norm = (T1_curr - mins_t[5]) / ranges_t[5]

    return torch.stack([T0_phys_norm, T1_phys_norm], dim=1), phys


# ==========================================
# 5. LOAD MODEL & RUN TEST
# ==========================================
print(f"\n[MODEL] Loading best checkpoint from: {str(BEST_PATH).replace(prefix, '')}")
model = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES).to(device)

checkpoint = torch.load(BEST_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
best_epoch = checkpoint['epoch']
best_val_loss = checkpoint['best_val_loss']

model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"[MODEL] Loaded from Epoch {best_epoch} | Best Val Loss: {best_val_loss:.6f}")
print(f"[MODEL] Total Parameters: {total_params:,}")

# --- Print learned physics parameters ---
print(f"\n[PHYSICS] ========== LEARNED PHYSICS PARAMETERS ==========")
learned_phys = model.get_bounded_physics()
for k in DEFAULT_EMPIRICAL:
    raw = model.raw_params[k].item()
    learned = learned_phys[k].item()
    empirical = DEFAULT_EMPIRICAL[k]
    pct = ((learned / empirical) - 1) * 100
    if k in ('q0', 'q1'):
        print(f"[PHYSICS]   {k:>5s}: learned={learned:.6f} | empirical={empirical:.6f} | Δ={pct:+.2f}% | (softplus)")
    else:
        print(f"[PHYSICS]   {k:>5s}: learned={learned:.6f} | empirical={empirical:.6f} | Δ={pct:+.2f}% | (sigmoid)")
print(f"[PHYSICS] =====================================================")

# --- Load test shards ---
test_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "test").glob("*.tar"))]
print(f"\n[DATA] Test shards: {len(test_files)} files")
if not test_files:
    print(f"[FATAL] No test shards found in: {LOCAL_SHARDS_DIR / 'test'}")
    sys.exit(1)
print(f"[DATA]   First test shard: {Path(test_files[0]).name}")
print(f"[DATA]   Last  test shard: {Path(test_files[-1]).name}")

test_loader = create_test_dataloader(test_files)

# ==========================================
# 6. EVALUATION LOOP
# ==========================================
mse_loss = nn.MSELoss()

# Accumulators for normalized-space losses
total_test_loss_data = 0.0
total_test_loss_phys = 0.0
num_test_batches = 0
total_test_samples = 0

# Accumulators for absolute-space (°C) metrics
sum_abs_err_T0 = 0.0   # Sum of |pred - true| for GPU0 temp (°C)
sum_abs_err_T1 = 0.0   # Sum of |pred - true| for GPU1 temp (°C)
sum_sq_err_T0 = 0.0    # Sum of (pred - true)² for GPU0 temp (°C²)
sum_sq_err_T1 = 0.0    # Sum of (pred - true)² for GPU1 temp (°C²)
max_abs_err_T0 = 0.0   # Max |pred - true| for GPU0 temp (°C)
max_abs_err_T1 = 0.0   # Max |pred - true| for GPU1 temp (°C)

# Accumulators for physics absolute-space metrics
sum_abs_err_phys_T0 = 0.0
sum_abs_err_phys_T1 = 0.0
sum_sq_err_phys_T0 = 0.0
sum_sq_err_phys_T1 = 0.0

print(f"\n{'='*80}")
print(f"[TEST] RUNNING FULL TEST EVALUATION [{VERSION_PREFIX}]")
print(f"[TEST] Model from Epoch {best_epoch} | Device: {device}")
print(f"{'='*80}")

test_start_time = time.time()

with torch.no_grad():
    test_pbar = tqdm(test_loader, desc="Testing", leave=True)

    for batch_idx, data_block in enumerate(test_pbar):
        data_block = data_block.to(device, non_blocking=True)
        batch_size_actual = data_block.shape[0]

        # --- Slice & normalize (same as training) ---
        X_seq = data_block[:, :50, :]
        Y_true_abs = data_block[:, -1, 4:6]   # [B, 2] absolute °C
        Y_true_norm = (Y_true_abs - mins[4:6]) / ranges[4:6]
        X_norm = (X_seq - mins) / ranges

        # --- TCN forward pass ---
        with torch.amp.autocast('cuda'):
            Y_pred_norm = model(X_norm)

        # --- Data loss (normalized) ---
        loss_data = mse_loss(Y_pred_norm, Y_true_norm)

        # --- Physics loss (normalized) ---
        T_phys_norm, _ = compute_physics_targets_norm(model, data_block, DT, mins, ranges)
        loss_phys = mse_loss(Y_pred_norm.float(), T_phys_norm)

        # --- Convert predictions to absolute °C ---
        Y_pred_abs = Y_pred_norm.float() * ranges[4:6] + mins[4:6]  # [B, 2]
        T_phys_abs = T_phys_norm * ranges[4:6] + mins[4:6]          # [B, 2]

        # --- Absolute-space errors (°C) ---
        abs_err = (Y_pred_abs - Y_true_abs).abs()      # [B, 2]
        sq_err = (Y_pred_abs - Y_true_abs).pow(2)      # [B, 2]

        abs_err_phys = (T_phys_abs - Y_true_abs).abs()  # [B, 2]
        sq_err_phys = (T_phys_abs - Y_true_abs).pow(2)  # [B, 2]

        # --- Accumulate ---
        total_test_loss_data += loss_data.item()
        total_test_loss_phys += loss_phys.item()
        num_test_batches += 1
        total_test_samples += batch_size_actual

        sum_abs_err_T0 += abs_err[:, 0].sum().item()
        sum_abs_err_T1 += abs_err[:, 1].sum().item()
        sum_sq_err_T0 += sq_err[:, 0].sum().item()
        sum_sq_err_T1 += sq_err[:, 1].sum().item()
        max_abs_err_T0 = max(max_abs_err_T0, abs_err[:, 0].max().item())
        max_abs_err_T1 = max(max_abs_err_T1, abs_err[:, 1].max().item())

        sum_abs_err_phys_T0 += abs_err_phys[:, 0].sum().item()
        sum_abs_err_phys_T1 += abs_err_phys[:, 1].sum().item()
        sum_sq_err_phys_T0 += sq_err_phys[:, 0].sum().item()
        sum_sq_err_phys_T1 += sq_err_phys[:, 1].sum().item()

        test_pbar.set_postfix({
            "Ld": f"{loss_data.item():.2e}",
            "Lp": f"{loss_phys.item():.2e}",
            "N": f"{total_test_samples:,}"
        })

test_duration = time.time() - test_start_time

# ==========================================
# 7. COMPUTE FINAL METRICS
# ==========================================
avg_test_loss_data = total_test_loss_data / num_test_batches
avg_test_loss_phys = total_test_loss_phys / num_test_batches

# TCN (neural network) metrics
mae_T0 = sum_abs_err_T0 / total_test_samples      # °C
mae_T1 = sum_abs_err_T1 / total_test_samples      # °C
mse_T0 = sum_sq_err_T0 / total_test_samples       # °C²
mse_T1 = sum_sq_err_T1 / total_test_samples       # °C²
rmse_T0 = mse_T0 ** 0.5                           # °C
rmse_T1 = mse_T1 ** 0.5                           # °C

# Physics (ODE integration) metrics
mae_phys_T0 = sum_abs_err_phys_T0 / total_test_samples
mae_phys_T1 = sum_abs_err_phys_T1 / total_test_samples
mse_phys_T0 = sum_sq_err_phys_T0 / total_test_samples
mse_phys_T1 = sum_sq_err_phys_T1 / total_test_samples
rmse_phys_T0 = mse_phys_T0 ** 0.5
rmse_phys_T1 = mse_phys_T1 ** 0.5

# ==========================================
# 8. PRINT RESULTS
# ==========================================
SEP = "=" * 80
print(f"\n{SEP}")
print(f"[RESULTS] COMPLETE TEST EVALUATION RESULTS [{VERSION_PREFIX}]")
print(f"{SEP}")
print(f"[RESULTS] Model loaded from: Epoch {best_epoch}")
print(f"[RESULTS] Best validation loss (during training): {best_val_loss:.6f}")
print(f"[RESULTS] Total test samples evaluated: {total_test_samples:,}")
print(f"[RESULTS] Total test batches: {num_test_batches}")
print(f"[RESULTS] Evaluation time: {test_duration:.1f}s")

print(f"\n[RESULTS] --- NORMALIZED-SPACE LOSSES (MSE in [0,1]) ---")
print(f"[RESULTS]   Avg Data Loss  : {avg_test_loss_data:.8f}")
print(f"[RESULTS]   Avg Physics Loss: {avg_test_loss_phys:.8f}")
print(f"[RESULTS]   RATIO (phys/data): {avg_test_loss_phys / max(avg_test_loss_data, 1e-12):.2f}x")

print(f"\n[RESULTS] --- TCN PREDICTION ACCURACY (Absolute °C) ---")
print(f"[RESULTS]   {'Metric':<20s} | {'GPU 0 (T0)':>12s} | {'GPU 1 (T1)':>12s} | {'Average':>12s}")
print(f"[RESULTS]   {'-'*20}-+-{'-'*12}-+-{'-'*12}-+-{'-'*12}")
print(f"[RESULTS]   {'MAE (°C)':<20s} | {mae_T0:>12.4f} | {mae_T1:>12.4f} | {(mae_T0 + mae_T1) / 2:>12.4f}")
print(f"[RESULTS]   {'MSE (°C²)':<20s} | {mse_T0:>12.6f} | {mse_T1:>12.6f} | {(mse_T0 + mse_T1) / 2:>12.6f}")
print(f"[RESULTS]   {'RMSE (°C)':<20s} | {rmse_T0:>12.4f} | {rmse_T1:>12.4f} | {(rmse_T0 + rmse_T1) / 2:>12.4f}")
print(f"[RESULTS]   {'Max Abs Error (°C)':<20s} | {max_abs_err_T0:>12.4f} | {max_abs_err_T1:>12.4f} | {max(max_abs_err_T0, max_abs_err_T1):>12.4f}")

print(f"\n[RESULTS] --- PHYSICS (ODE) PREDICTION ACCURACY (Absolute °C) ---")
print(f"[RESULTS]   {'Metric':<20s} | {'GPU 0 (T0)':>12s} | {'GPU 1 (T1)':>12s} | {'Average':>12s}")
print(f"[RESULTS]   {'-'*20}-+-{'-'*12}-+-{'-'*12}-+-{'-'*12}")
print(f"[RESULTS]   {'MAE (°C)':<20s} | {mae_phys_T0:>12.4f} | {mae_phys_T1:>12.4f} | {(mae_phys_T0 + mae_phys_T1) / 2:>12.4f}")
print(f"[RESULTS]   {'MSE (°C²)':<20s} | {mse_phys_T0:>12.6f} | {mse_phys_T1:>12.6f} | {(mse_phys_T0 + mse_phys_T1) / 2:>12.6f}")
print(f"[RESULTS]   {'RMSE (°C)':<20s} | {rmse_phys_T0:>12.4f} | {rmse_phys_T1:>12.4f} | {(rmse_phys_T0 + rmse_phys_T1) / 2:>12.4f}")

print(f"\n[RESULTS] --- LEARNED PHYSICS PARAMETERS ---")
for k in DEFAULT_EMPIRICAL:
    learned = learned_phys[k].item()
    empirical = DEFAULT_EMPIRICAL[k]
    pct = ((learned / empirical) - 1) * 100
    print(f"[RESULTS]   {k:>5s} = {learned:.6f}  (empirical={empirical:.6f}, Δ={pct:+.2f}%)")

print(f"\n{SEP}")
print(f"[RESULTS] INTERPRETATION GUIDE:")
print(f"[RESULTS]   MAE < 0.5°C  → Excellent")
print(f"[RESULTS]   MAE < 1.0°C  → Good")
print(f"[RESULTS]   MAE < 2.0°C  → Acceptable")
print(f"[RESULTS]   MAE > 2.0°C  → Needs improvement")
print(f"[RESULTS]   Physics/Data loss ratio ~1x → Well-balanced PINN")
print(f"{SEP}")

# ==========================================
# 9. SAVE RESULTS TO CSV
# ==========================================
test_results_path = SAVE_DIR / f"{VERSION_PREFIX}_test_results.csv"
with open(test_results_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Metric', 'GPU_0_T0', 'GPU_1_T1', 'Average'])
    writer.writerow(['TCN_MAE_C', f'{mae_T0:.6f}', f'{mae_T1:.6f}', f'{(mae_T0 + mae_T1) / 2:.6f}'])
    writer.writerow(['TCN_MSE_C2', f'{mse_T0:.6f}', f'{mse_T1:.6f}', f'{(mse_T0 + mse_T1) / 2:.6f}'])
    writer.writerow(['TCN_RMSE_C', f'{rmse_T0:.6f}', f'{rmse_T1:.6f}', f'{(rmse_T0 + rmse_T1) / 2:.6f}'])
    writer.writerow(['TCN_MaxErr_C', f'{max_abs_err_T0:.6f}', f'{max_abs_err_T1:.6f}', f'{max(max_abs_err_T0, max_abs_err_T1):.6f}'])
    writer.writerow(['Phys_MAE_C', f'{mae_phys_T0:.6f}', f'{mae_phys_T1:.6f}', f'{(mae_phys_T0 + mae_phys_T1) / 2:.6f}'])
    writer.writerow(['Phys_MSE_C2', f'{mse_phys_T0:.6f}', f'{mse_phys_T1:.6f}', f'{(mse_phys_T0 + mse_phys_T1) / 2:.6f}'])
    writer.writerow(['Phys_RMSE_C', f'{rmse_phys_T0:.6f}', f'{rmse_phys_T1:.6f}', f'{(rmse_phys_T0 + rmse_phys_T1) / 2:.6f}'])
    writer.writerow(['Norm_Data_Loss', f'{avg_test_loss_data:.8f}', '', ''])
    writer.writerow(['Norm_Phys_Loss', f'{avg_test_loss_phys:.8f}', '', ''])
    writer.writerow(['Total_Samples', f'{total_test_samples}', '', ''])
    writer.writerow(['Best_Epoch', f'{best_epoch}', '', ''])
    writer.writerow(['Best_Val_Loss', f'{best_val_loss:.8f}', '', ''])

    # Physics parameters
    writer.writerow([])
    writer.writerow(['Physics_Param', 'Learned', 'Empirical', 'Delta_Pct'])
    for k in DEFAULT_EMPIRICAL:
        learned = learned_phys[k].item()
        empirical = DEFAULT_EMPIRICAL[k]
        pct = ((learned / empirical) - 1) * 100
        writer.writerow([k, f'{learned:.6f}', f'{empirical:.6f}', f'{pct:+.2f}%'])

print(f"\n[SYSTEM] Test results saved to: {str(test_results_path).replace(prefix, '')}")
print(f"[SYSTEM] Test evaluation complete.")



[SYSTEM] Live GPU Status:
Fri Mar 27 19:23:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 55%   67C    P8             21W /  200W |    9391MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  0


[SYSTEM] Manually locking script to GPU 0.
[SYSTEM] PyTorch initialized. Active Device: NVIDIA RTX A4500 (cuda:0)
[SYSTEM] CUDA Version: 12.6 | PyTorch Version: 2.9.1+cu126
[SYSTEM] GPU Memory: 19.6 GB



Enter version prefix to TEST (e.g., TCN_ReLoBRaLo_v3) [Press Enter for default 'unknown_version']:  TCN_ReLoBRaLo_v3



[PATHS] PROJECT_DIR     : Capstone/Implementation
[PATHS] LOCAL_SHARDS_DIR: Capstone/Implementation/data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards
[PATHS] SAVE_DIR        : Capstone/Implementation/models/TCN_ReLoBRaLo_v3
[PATHS] BEST_PATH       : Capstone/Implementation/models/TCN_ReLoBRaLo_v3/TCN_ReLoBRaLo_v3_best_model.pt

[DATA] Normalization scalars loaded successfully.
[DATA] Feature Order: ['power_draw_gpu_0_W', 'power_draw_gpu_1_W', 'utilization_gpu_0_pct', 'utilization_gpu_1_pct', 'temperature_gpu_0', 'temperature_gpu_1']
[DATA] Number of Features: 6
[DATA]   [0]             power_draw_gpu_0_W | min=     21.8100 | max=    331.0800
[DATA]   [1]             power_draw_gpu_1_W | min=     22.7800 | max=    328.8600
[DATA]   [2]          utilization_gpu_0_pct | min=      0.0000 | max=    100.0000
[DATA]   [3]          utilization_gpu_1_pct | min=      0.0000 | max=    100.0000
[DATA]   [4]              temperature_gpu_0 | min=     23.0000

Testing: 1359it [01:36, 14.02it/s, Ld=8.21e-04, Lp=4.55e-04, N=22,184,780]


[RESULTS] COMPLETE TEST EVALUATION RESULTS [TCN_ReLoBRaLo_v3]
[RESULTS] Model loaded from: Epoch 1
[RESULTS] Best validation loss (during training): 0.002016
[RESULTS] Total test samples evaluated: 22,184,780
[RESULTS] Total test batches: 1359
[RESULTS] Evaluation time: 96.9s

[RESULTS] --- NORMALIZED-SPACE LOSSES (MSE in [0,1]) ---
[RESULTS]   Avg Data Loss  : 0.00092368
[RESULTS]   Avg Physics Loss: 0.00070936
[RESULTS]   RATIO (phys/data): 0.77x

[RESULTS] --- TCN PREDICTION ACCURACY (Absolute °C) ---
[RESULTS]   Metric               |   GPU 0 (T0) |   GPU 1 (T1) |      Average
[RESULTS]   ---------------------+--------------+--------------+-------------
[RESULTS]   MAE (°C)             |       1.2848 |       1.4813 |       1.3830
[RESULTS]   MSE (°C²)            |     2.969089 |     3.923721 |     3.446405
[RESULTS]   RMSE (°C)            |       1.7231 |       1.9808 |       1.8520
[RESULTS]   Max Abs Error (°C)   |      16.1680 |      16.5967 |      16.5967

[RESULTS] --- PHYSIC

# 